# 1.1 Foundational Models - Gemini

This notebook uses Gemini through LangChain. API keys stay in `.env` and are never written in notebook cells.

In [ ]:
from pprint import pprint

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

load_dotenv()

## Initialise and invoke a model

The response is an object: inspect it before extracting its text.

In [ ]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.2,
    max_output_tokens=256,
    timeout=30,
    max_retries=2,
)

response = model.invoke("What is the capital of the Moon? Answer in one sentence.")
response

In [ ]:
print(response.content)
print("\nResponse metadata:")
pprint(response.response_metadata)
print("\nUsage metadata:")
pprint(response.usage_metadata)

## Change one dial at a time

Temperature changes randomness; `max_output_tokens`, `timeout`, and `max_retries` are set above for concise, resilient calls.

In [ ]:
question = "Invent a name for a city on the Moon and explain it in one sentence."

for label, temperature in [("Low temperature", 0.1), ("High temperature", 1.0)]:
    trial_model = ChatGoogleGenerativeAI(
        model="gemini-3.1-flash-lite",
        temperature=temperature,
        max_output_tokens=128,
        timeout=30,
        max_retries=2,
    )
    print(f"{label}:")
    print(trial_model.invoke(question).content)
    print()

## System prompt and examples

The system instruction defines the role. The two example pairs make the expected response format concrete.

In [ ]:
prompted_response = model.invoke([
    SystemMessage(content="You are a science-fiction city archivist. Reply with exactly one evocative sentence."),
    HumanMessage(content="Describe a lunar library."),
    AIMessage(content="The Selenic Archive stores its books in silent magnetic vaults beneath Mare Tranquillitatis."),
    HumanMessage(content="Describe a lunar market."),
    AIMessage(content="At Apollo Bazaar, traders exchange ice credits beneath silver dust lanterns."),
    HumanMessage(content="Describe a lunar train station."),
])

print(prompted_response.content)

## Create an agent

An agent returns the full message history. The final item is its answer.

In [ ]:
agent = create_agent(
    model=model,
    system_prompt="You are a helpful science-fiction guide. Be imaginative but clearly label invented facts as fiction.",
)

agent_response = agent.invoke({
    "messages": [HumanMessage(content="What is the capital of the Moon?")]
})

pprint(agent_response)
print("\nFinal answer:")
print(agent_response["messages"][-1].content)

In [ ]:
conversation_response = agent.invoke({
    "messages": [
        HumanMessage(content="What is the capital of the Moon?"),
        AIMessage(content="In this fictional setting, the capital is Luna City."),
        HumanMessage(content="Tell me one detail about its transport system."),
    ]
})

print(conversation_response["messages"][-1].content)

## Stream the answer

Streaming does not reduce the model's work, but it makes progress visible immediately.

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Describe Luna City in three short sentences.")]},
    stream_mode="messages",
):
    if token.content:
        print(token.content, end="", flush=True)

print()

## Ask for structured output

A schema gives code stable fields instead of text that needs to be parsed.

In [ ]:
class LunarPlace(BaseModel):
    name: str = Field(description="Invented place name")
    purpose: str = Field(description="What the place is used for")
    atmosphere: str = Field(description="A concise mood description")

structured_model = model.with_structured_output(LunarPlace)
lunar_place = structured_model.invoke("Invent one memorable place in Luna City.")
lunar_place